# Action Distribution Plots

Lean notebook backed by `action_distribution_metrics.py`. The original large notebook remains at `../action_distribution_plots.ipynb`.


In [ ]:
from pathlib import Path
import importlib
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    local_metrics = candidate / "action_distribution_metrics.py"
    repo_metrics = candidate / "Topology_Task" / "configs" / "zz_print_metrics" / "action_distribution_metrics.py"
    if local_metrics.exists():
        sys.path.insert(0, str(candidate))
        break
    if repo_metrics.exists():
        sys.path.insert(0, str(repo_metrics.parent))
        break

import action_distribution_metrics as adm
adm = importlib.reload(adm)
import wandb_metrics as wm
wm = importlib.reload(wm)
print("action_distribution_metrics:", adm.__file__)
print("wandb_metrics:", wm.__file__)


## Choose Runs To Load Or Download


In [ ]:
import importlib
wm = importlib.reload(wm)

# Pick config folder(s) under Topology_Task/configs, or use None / "None" / "all" for every run.
# Examples:
# CONFIG_FOLDERS_TO_DOWNLOAD = "intervention_gate_15"
# CONFIG_FOLDERS_TO_DOWNLOAD = ["intervention_gate_15", "phase4_sparse_control_16"]
# CONFIG_FOLDERS_TO_DOWNLOAD = "intervention_gate_15,phase4_sparse_control_16"
# CONFIG_FOLDERS_TO_DOWNLOAD = "all"
CONFIG_FOLDERS_TO_DOWNLOAD = [
    "intervention_gate_15",
    "phase4_sparse_control_16",
    "heuristic_vs_gate_s0_s1_s2",
]

# False only reads local cached histories. True downloads missing matching histories from W&B.
DOWNLOAD_MISSING_FROM_WANDB = True

# Useful when W&B has newer data than the local cache, especially for old scan_history fallback caches.
# This only has an effect when DOWNLOAD_MISSING_FROM_WANDB is True.
REFRESH_SCAN_HISTORY_FALLBACKS = True

# Heavier option: replace every selected local cache from W&B.
FORCE_REFRESH_CACHE = False

wm.configure_run_filter_from_config_folder(CONFIG_FOLDERS_TO_DOWNLOAD)
wm.EXCLUDE_RUN_NAME_REGEX = None
wm.RUN_STATES = None
wm.MAX_RUNS = None
wm.USE_LOCAL_CACHE_ONLY = not DOWNLOAD_MISSING_FROM_WANDB
wm.REFRESH_SCAN_HISTORY_FALLBACKS = bool(DOWNLOAD_MISSING_FROM_WANDB and REFRESH_SCAN_HISTORY_FALLBACKS)
wm.FORCE_REFRESH = bool(DOWNLOAD_MISSING_FROM_WANDB and FORCE_REFRESH_CACHE)
wm.ALLOW_SCAN_HISTORY_FALLBACK = True
wm.refresh_run_filters()

print("RUN_NAME_REGEX =", wm.RUN_NAME_REGEX)
print("USE_LOCAL_CACHE_ONLY =", wm.USE_LOCAL_CACHE_ONLY)
print("REFRESH_SCAN_HISTORY_FALLBACKS =", wm.REFRESH_SCAN_HISTORY_FALLBACKS)
print("FORCE_REFRESH =", wm.FORCE_REFRESH)


## Load Or Download Selected W&B Histories


In [ ]:
download_data = wm.load_wandb_data()
downloaded_runs_df = download_data.runs_df
print(f"Matching W&B runs: {len(downloaded_runs_df)}")


## Load Cached Histories And Shared Tables


In [ ]:
# Use final evaluation action metrics for the action-distribution plots.
# Set ACTION_METRIC_SOURCE = "train" to reproduce the old rollout-training plots.
ACTION_METRIC_SOURCE = "eval"
EVAL_METRIC_SPLIT = "test"

# Final-window plots average the last logged points at or before this step.
# Use None to treat each run's actual last logged point as final.
FINAL_SUMMARY_STEP_M = 11.0

ctx = adm.load_action_distribution_context(
    action_metric_source=ACTION_METRIC_SOURCE,
    eval_split=EVAL_METRIC_SPLIT,
    final_summary_step_m=FINAL_SUMMARY_STEP_M,
    save_figures=True,
    show_figures=False,
)


## Last-5logged action 0 fraction by agent for all runs


In [ ]:
last5_action0 = adm.last5_logged_action0_fraction_by_agent_all_runs(ctx)
adm.display_metric_figures(last5_action0)


## Seed-Aggregated Last-5-Logged Action-0 Fraction By Agent


In [ ]:
seed_action0 = adm.seed_aggregated_last5_logged_action0_fraction_by_agent(ctx)
adm.display_metric_figures(seed_action0)


## Survival vs Action-0 / Non-Idle Over Time


In [ ]:
survival_action = adm.survival_vs_action0_non_idle_over_time(ctx)
adm.display_metric_figures(survival_action)


## Entropy Collapse vs Action-0 Confidence


In [ ]:
entropy_action0 = adm.entropy_collapse_vs_action0_confidence(ctx)
adm.display_metric_figures(entropy_action0)


## Agent Non-Idle Imbalance


In [ ]:
agent_imbalance = adm.agent_non_idle_imbalance(ctx)
adm.display_metric_figures(agent_imbalance)


## Joint Action Coordination


In [ ]:
joint_coordination = adm.joint_action_coordination(ctx)
adm.display_metric_figures(joint_coordination)


## Gate Probability vs Actual Intervention Fraction


In [ ]:
gate_calibration = adm.gate_probability_vs_actual_intervention_fraction(ctx)
adm.display_metric_figures(gate_calibration)
